# Average U,V in sigma layers

TO calcualte the EKE, first we will need to calculate the 3 years average U and V in simulations. This notebook will do that

In [6]:
import cosima_cookbook as cc
import matplotlib.pyplot as plt
import cmocean as cm
import numpy as np
from dask.distributed import Client
from scipy.interpolate import interp1d

import xarray as xr
import cf_xarray as cfxr

In [7]:
client = Client(memory_limit = '1400gb',n_workers = 48)

2025-01-29 11:21:56,128 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:37787' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('concatenate-open_dataset-volcello-getitem-ed4edf1f5578c635a9e56ede09340381', 103, 0, 4, 1), ('concatenate-open_dataset-volcello-getitem-ed4edf1f5578c635a9e56ede09340381', 103, 2, 2, 70), ('concatenate-open_dataset-volcello-getitem-ed4edf1f5578c635a9e56ede09340381', 103, 0, 3, 42), ('concatenate-open_dataset-volcello-getitem-ed4edf1f5578c635a9e56ede09340381', 103, 3, 10, 42), ('concatenate-open_dataset-volcello-getitem-ed4edf1f5578c635a9e56ede09340381', 103, 1, 3, 66), ('concatenate-open_dataset-volcello-getitem-ed4edf1f5578c635a9e56ede09340381', 103, 0, 6, 47), ('getitem-ed4edf1f5578c635a9e56ede09340381', 103, 2, 8, 12), ('getitem-ed4edf1f5578c635a9e56ede09340381', 103, 1, 6, 26), ('concatenate-open_dataset-volcello-getitem-ed4edf1f5578c635a9e56ede09340381', 103, 2, 13, 84), ('getitem-ed4edf1f5578

## First for panan01

In [8]:
session = cc.database.create_session('/home/156/wf4500/databases/access/panan01_HTsigma.db') #panan01
exp = 'panan01_yr2000_HTsigma'

start_time = '1991-01-01'
end_time = '2000-12-31'
lat_range = slice(-90,-59)


In [6]:
#importing volume
vol_centre = cc.querying.getvar(exp,'volcello',session,ncfile='%month_rho2.nc',start_time=start_time,end_time=end_time,chunks={})\
.sel(time=slice(start_time,end_time)).sel(yh=lat_range).mean('time').compute()

In [7]:
#Getting mass transport in kg/s
vmo_nt = cc.querying.getvar(exp,'vmo',session,ncfile='%month_rho%',start_time=start_time,end_time=end_time,chunks={}).sel(time=slice(start_time,end_time))
umo_et = cc.querying.getvar(exp,'umo',session,ncfile='%month_rho%',start_time=start_time,end_time=end_time,chunks={}).sel(time=slice(start_time,end_time))

vmo_nt = vmo_nt.sel(yq=lat_range).sel(time=slice(start_time,end_time)).mean('time').compute()
umo_et = umo_et.sel(yh=lat_range).sel(time=slice(start_time,end_time)).mean('time').compute()

In [8]:
#Making volume on xq and yq grid  ######

#First making a halo for proper edges interpolation
lon_interval = vol_centre.xh.diff('xh').mean()
lon_rightappend = vol_centre.xh[-1].values+np.cumsum([lon_interval,lon_interval,lon_interval,lon_interval,lon_interval])
lon_leftappend = vol_centre.xh[0].values - \
np.flip(np.cumsum([lon_interval,lon_interval,lon_interval,lon_interval]))

newlon = np.concatenate((lon_leftappend,vol_centre.xh.values,lon_rightappend), axis=0)
vol_centre_halo = xr.concat([vol_centre.isel(xh=slice(-5,-1)),vol_centre,vol_centre.isel(xh=slice(0,5))], dim='xh')
vol_centre_halo['xh'] = newlon


#Now interpolating volumes onto the xq and yq grids
vol_yhxq = vol_centre_halo.interp(xh=umo_et.xq)
vol_yqxq = vol_centre_halo.interp(xh=umo_et.xq,yh=vmo_nt.yq)
vol_yqxh = vol_centre_halo.interp(yh=vmo_nt.yq)
vol_yhxh = vol_centre.copy()

In [9]:
#dxs in all grid points
dx_xhyh = cc.querying.getvar(exp,'dxt',session,ncfile='19990101.ocean_static.nc')
dx_xqyh = cc.querying.getvar(exp,'dxCu',session,ncfile='19990101.ocean_static.nc')
dx_xhyq = cc.querying.getvar(exp,'dxCv',session,ncfile='19990101.ocean_static.nc')
dx_xqyq = dx_xhyh.rename({'xh':'xq','yh':'yq'}).copy()
dx_xqyq=  xr.concat([dx_xqyq[:,-1],dx_xqyq], dim='xq')
dx_xqyq['yq'] = dx_xhyq.yq[:-1]
dx_xqyq['xq'] = dx_xqyh.xq

#dys in all grid points
dy_xhyh = cc.querying.getvar(exp,'dyt',session,ncfile='19990101.ocean_static.nc')
dy_xqyh = cc.querying.getvar(exp,'dyCu',session,ncfile='19990101.ocean_static.nc')
dy_xhyq = cc.querying.getvar(exp,'dyCv',session,ncfile='19990101.ocean_static.nc')
dy_xqyq = dy_xhyh.rename({'xh':'xq','yh':'yq'}).copy()
dy_xqyq=  xr.concat([dy_xqyq[:,-1],dy_xqyq], dim='xq')
dy_xqyq['yq'] = dy_xhyq.yq[:-1]
dy_xqyq['xq'] = dy_xqyh.xq

#setting the reference rho
rho_ref = 1035 #kg m^{-3}, used as standard by Pan-Antarctic

Calcualting U and V in the rights spots for future EKE calcualtion

In [10]:
#first for U
U_xq = (umo_et*(1/rho_ref) * (1/vol_yhxq) * dx_xqyh).compute()

#now for V
V_yq = (vmo_nt*(1/rho_ref) * (1/vol_yqxh) * dy_xhyq).compute()

In [11]:
U_xq = U_xq.drop_vars('xh').where(U_xq>-9e10).where(U_xq<9e10)
V_yq  = V_yq .drop_vars('yh').where(V_yq>-9e10).where(V_yq<9e10)

saving these 3 years means

In [12]:
U_xq.name='uo'
V_yq.name='vo'

In [13]:
#saving file
dir_file = '/g/data/ik11/users/wf4500/Project_panan/GH/Panan_HT_ASC/Processed_data/panan01_rerun/EKE_along_contour_rho/'
Uname_file = dir_file + 'U_mean_1991_2000.nc'
U_xq.to_netcdf(Uname_file)

Vname_file = dir_file + 'V_mean_1991_2000.nc'
V_yq.to_netcdf(Vname_file)

## Now for panan005

In [4]:
session = cc.database.create_session('/home/156/wf4500/databases/access/panan005_rerun.db')
exp = 'panan_005deg_jra55_ryf_2024_12_14'


In [5]:
#importing volume
vol_centre = cc.querying.getvar(exp,'volcello',session,ncfile='%month_rho2.nc',start_time=start_time,end_time=end_time,chunks={})\
.sel(time=slice(start_time,end_time)).sel(yh=lat_range).mean('time').compute()

In [6]:
#Getting mass transport in kg/s
vmo_nt = cc.querying.getvar(exp,'vmo',session,ncfile='%month_rho%',start_time=start_time,end_time=end_time,chunks={}).sel(time=slice(start_time,end_time))
umo_et = cc.querying.getvar(exp,'umo',session,ncfile='%month_rho%',start_time=start_time,end_time=end_time,chunks={}).sel(time=slice(start_time,end_time))

vmo_nt = vmo_nt.sel(yq=lat_range).sel(time=slice(start_time,end_time)).mean('time').compute()
umo_et = umo_et.sel(yh=lat_range).sel(time=slice(start_time,end_time)).mean('time').compute()

In [7]:
#Making volume on xq and yq grid  ######

#First making a halo for proper edges interpolation
lon_interval = vol_centre.xh.diff('xh').mean()
lon_rightappend = vol_centre.xh[-1].values+np.cumsum([lon_interval,lon_interval,lon_interval,lon_interval,lon_interval])
lon_leftappend = vol_centre.xh[0].values - \
np.flip(np.cumsum([lon_interval,lon_interval,lon_interval,lon_interval]))

newlon = np.concatenate((lon_leftappend,vol_centre.xh.values,lon_rightappend), axis=0)
vol_centre_halo = xr.concat([vol_centre.isel(xh=slice(-5,-1)),vol_centre,vol_centre.isel(xh=slice(0,5))], dim='xh')
vol_centre_halo['xh'] = newlon


#Now interpolating volumes onto the xq and yq grids
vol_yhxq = vol_centre_halo.interp(xh=umo_et.xq)
vol_yqxq = vol_centre_halo.interp(xh=umo_et.xq,yh=vmo_nt.yq)
vol_yqxh = vol_centre_halo.interp(yh=vmo_nt.yq)
vol_yhxh = vol_centre.copy()

In [8]:
#dxs in all grid points
dx_xhyh = cc.querying.getvar(exp,'dxt',session,ncfile='19990101.ocean_static.nc').compute()
dx_xqyh = cc.querying.getvar(exp,'dxCu',session,ncfile='19990101.ocean_static.nc').compute()
dx_xhyq = cc.querying.getvar(exp,'dxCv',session,ncfile='19990101.ocean_static.nc').compute()
dx_xqyq = dx_xhyh.rename({'xh':'xq','yh':'yq'}).copy()
dx_xqyq=  xr.concat([dx_xqyq[:,-1],dx_xqyq], dim='xq')
dx_xqyq['yq'] = dx_xhyq.yq[:-1]
dx_xqyq['xq'] = dx_xqyh.xq

#dys in all grid points
dy_xhyh = cc.querying.getvar(exp,'dyt',session,ncfile='19990101.ocean_static.nc').compute()
dy_xqyh = cc.querying.getvar(exp,'dyCu',session,ncfile='19990101.ocean_static.nc').compute()
dy_xhyq = cc.querying.getvar(exp,'dyCv',session,ncfile='19990101.ocean_static.nc').compute()
dy_xqyq = dy_xhyh.rename({'xh':'xq','yh':'yq'}).copy()
dy_xqyq=  xr.concat([dy_xqyq[:,-1],dy_xqyq], dim='xq')
dy_xqyq['yq'] = dy_xhyq.yq[:-1]
dy_xqyq['xq'] = dy_xqyh.xq

#setting the reference rho
rho_ref = 1035 #kg m^{-3}, used as standard by Pan-Antarctic

In [10]:
#first for U
U_xq = (umo_et*(1/rho_ref) * (1/vol_yhxq) * dx_xqyh).compute()

#now for V
V_yq = (vmo_nt*(1/rho_ref) * (1/vol_yqxh) * dy_xhyq).compute()

In [11]:
U_xq = U_xq.drop_vars('xh').where(U_xq>-9e9999).where(U_xq<99e9999)
V_yq  = V_yq .drop_vars('yh').where(V_yq>-9e9999).where(V_yq<99e9999)

In [12]:
U_xq.name='uo'
V_yq.name='vo'

In [13]:
#saving file
dir_file = '/g/data/ik11/users/wf4500/Project_panan/GH/Panan_HT_ASC/Processed_data/panan005_rerun/EKE_along_contour_rho/'
Uname_file = dir_file + 'U_mean_1991_2000.nc'
U_xq.to_netcdf(Uname_file)

Vname_file = dir_file + 'V_mean_1991_2000.nc'
V_yq.to_netcdf(Vname_file)

## Now  for panan0025

Notice that panan0025 is substantially heavier than the other runs, so we have to do all this in parts here

In [9]:
session = cc.database.create_session('/home/156/wf4500/databases/access/panan0025_rerun.db') #panan0025
exp = 'mom6-panan'

start_time = '1991-01-01'
end_time = '2000-12-31'
lat_range = slice(-90,-59)

In [10]:
#importing volume
vol_centre = cc.querying.getvar(exp,'volcello',session,ncfile='%month_rho2.nc',start_time=start_time,end_time=end_time,chunks={})\
.sel(time=slice(start_time,end_time)).sel(yh=lat_range).mean('time').compute()

2025-01-29 11:20:25,247 - distributed.worker_memory - ERROR - [Errno 12] Cannot allocate memory
Traceback (most recent call last):
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/distributed/utils.py", line 837, in wrapper
    return await func(*args, **kwargs)
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/distributed/worker_memory.py", line 212, in memory_monitor
    memory = worker.monitor.get_process_memory()
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/distributed/system_monitor.py", line 150, in get_process_memory
    return self.proc.memory_info().rss
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/psutil/_common.py", line 497, in wrapper
    raise raise_from(err, None)
  File "<string>", line 3, in raise_from
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packa

KeyboardInterrupt: 

In [ ]:
#Getting mass transport in kg/s
vmo_nt = cc.querying.getvar(exp,'vmo',session,ncfile='%month_rho%',start_time=start_time,end_time=end_time,chunks={}).sel(time=slice(start_time,end_time))
umo_et = cc.querying.getvar(exp,'umo',session,ncfile='%month_rho%',start_time=start_time,end_time=end_time,chunks={}).sel(time=slice(start_time,end_time))

vmo_nt = vmo_nt.sel(yq=lat_range).sel(time=slice(start_time,end_time)).mean('time').compute()
umo_et = umo_et.sel(yh=lat_range).sel(time=slice(start_time,end_time)).mean('time').compute()

In [ ]:
#Making volume on xq and yq grid  ######

#First making a halo for proper edges interpolation
lon_interval = vol_centre.xh.diff('xh').mean()
lon_rightappend = vol_centre.xh[-1].values+np.cumsum([lon_interval,lon_interval,lon_interval,lon_interval,lon_interval])
lon_leftappend = vol_centre.xh[0].values - \
np.flip(np.cumsum([lon_interval,lon_interval,lon_interval,lon_interval]))

newlon = np.concatenate((lon_leftappend,vol_centre.xh.values,lon_rightappend), axis=0)
vol_centre_halo = xr.concat([vol_centre.isel(xh=slice(-5,-1)),vol_centre,vol_centre.isel(xh=slice(0,5))], dim='xh')
vol_centre_halo['xh'] = newlon


#Now interpolating volumes onto the xq and yq grids
vol_yhxq = vol_centre_halo.interp(xh=umo_et.xq)
vol_yqxq = vol_centre_halo.interp(xh=umo_et.xq,yh=vmo_nt.yq)
vol_yqxh = vol_centre_halo.interp(yh=vmo_nt.yq)
vol_yhxh = vol_centre.copy()

In [ ]:
#dxs in all grid points
dx_xhyh = cc.querying.getvar(exp,'dxt',session,ncfile='19990101.ocean_static.nc')
dx_xqyh = cc.querying.getvar(exp,'dxCu',session,ncfile='19990101.ocean_static.nc')
dx_xhyq = cc.querying.getvar(exp,'dxCv',session,ncfile='19990101.ocean_static.nc')
dx_xqyq = dx_xhyh.rename({'xh':'xq','yh':'yq'}).copy()
dx_xqyq=  xr.concat([dx_xqyq[:,-1],dx_xqyq], dim='xq')
dx_xqyq['yq'] = dx_xhyq.yq[:-1]
dx_xqyq['xq'] = dx_xqyh.xq

#dys in all grid points
dy_xhyh = cc.querying.getvar(exp,'dyt',session,ncfile='19990101.ocean_static.nc')
dy_xqyh = cc.querying.getvar(exp,'dyCu',session,ncfile='19990101.ocean_static.nc')
dy_xhyq = cc.querying.getvar(exp,'dyCv',session,ncfile='19990101.ocean_static.nc')
dy_xqyq = dy_xhyh.rename({'xh':'xq','yh':'yq'}).copy()
dy_xqyq=  xr.concat([dy_xqyq[:,-1],dy_xqyq], dim='xq')
dy_xqyq['yq'] = dy_xhyq.yq[:-1]
dy_xqyq['xq'] = dy_xqyh.xq

#setting the reference rho
rho_ref = 1035 #kg m^{-3}, used as standard by Pan-Antarctic

In [ ]:
#first for U
U_xq = (umo_et*(1/rho_ref) * (1/vol_yhxq) * dx_xqyh).compute()

#now for V
V_yq = (vmo_nt*(1/rho_ref) * (1/vol_yqxh) * dy_xhyq).compute()

In [ ]:
U_xq = U_xq.drop_vars('xh').where(U_xq>-9e10).where(U_xq<9e10)
V_yq  = V_yq .drop_vars('yh').where(V_yq>-9e10).where(V_yq<9e10)

In [ ]:
U_xq.name='uo'
V_yq.name='vo'

IOStream.flush timed out
2025-01-29 11:26:54,842 - tornado.application - ERROR - Exception in callback functools.partial(<function ZMQStream._update_handler.<locals>.<lambda> at 0x14b3df37f010>)
Traceback (most recent call last):
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/tornado/ioloop.py", line 750, in _run_callback
    ret = callback()
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py", line 695, in <lambda>
    self.io_loop.add_callback(lambda: self._handle_events(self.socket, 0))
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py", line 611, in _handle_events
    self._handle_recv()
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py", line 640, in _handle_recv
    self._run_callback(callback, msg)
  File "/g/data/hh5/public/app

In [ ]:
#saving file
dir_file = '/g/data/ik11/users/wf4500/Project_panan/GH/Panan_HT_ASC/Processed_data/panan0025_rerun/EKE_along_contour_rho/'
Uname_file = dir_file + 'U_mean_1991_2000.nc'
U_xq.to_netcdf(Uname_file)

Vname_file = dir_file + 'V_mean_1991_2000.nc'
V_yq.to_netcdf(Vname_file)